In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))  # so notebook can import from src/
from config.paths import RAW_DATA_PATH

df = pd.read_csv(RAW_DATA_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

print(df.shape)
print(df["date"].is_monotonic_increasing)
df.head(2)

(19735, 29)
True


,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.7900,19.79,44.73,19.0,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.7225,19.79,44.79,19.0,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195


In [2]:
LAG_STEPS = [1, 2, 3, 4, 5, 6, 144]

for lag in LAG_STEPS:
    df[f"Appliances_lag_{lag}"] = df["Appliances"].shift(lag)

df[["date", "Appliances", "Appliances_lag_1", "Appliances_lag_6", "Appliances_lag_144"]].head(3)

,date,Appliances,Appliances_lag_1,Appliances_lag_6,Appliances_lag_144
0,2016-01-11 17:00:00,60,NaN,NaN,NaN
1,2016-01-11 17:10:00,60,60.0,NaN,NaN
2,2016-01-11 17:20:00,50,60.0,NaN,NaN


In [3]:
# Deliberately broken: centered rolling window
df["Appliances_roll6_mean_LEAKY"] = df["Appliances"].rolling(window=6, center=True).mean()

# Correct: backward-only rolling window
df["Appliances_roll6_mean"] = df["Appliances"].rolling(window=6).mean()

# Inspect one row to SEE the leak concretely
row = 100
print("Target row date:", df.loc[row, "date"])
print("\nRows feeding the CENTERED (leaky) window:")
print(df.loc[row-2:row+3, ["date", "Appliances"]])  # center=True with window=6 uses ~[t-2, t+3]

print("\nLeaky mean :", df.loc[row, "Appliances_roll6_mean_LEAKY"])
print("Correct mean:", df.loc[row, "Appliances_roll6_mean"])


Target row date: 2016-01-12 09:40:00

Rows feeding the CENTERED (leaky) window:
                   date  Appliances
98  2016-01-12 09:20:00          50
99  2016-01-12 09:30:00          30
100 2016-01-12 09:40:00          40
101 2016-01-12 09:50:00          30
102 2016-01-12 10:00:00         260
103 2016-01-12 10:10:00         500

Leaky mean : 78.33333333333333
Correct mean: 48.333333333333336


In [4]:
ROLLING_WINDOWS = [6, 18]  # ~1hr, ~3hr

for window in ROLLING_WINDOWS:
    df[f"Appliances_roll{window}_mean"] = df["Appliances"].rolling(window=window).mean()
    df[f"Appliances_roll{window}_std"] = df["Appliances"].rolling(window=window).std()

# drop the leaky demo columns now that we've seen the failure — not part of the real feature set
df = df.drop(columns=["Appliances_roll6_mean_LEAKY"])

df[["date", "Appliances", "Appliances_roll6_mean", "Appliances_roll6_std",
    "Appliances_roll18_mean", "Appliances_roll18_std"]].head(20)

,date,Appliances,Appliances_roll6_mean,Appliances_roll6_std,Appliances_roll18_mean,Appliances_roll18_std
0,2016-01-11 17:00:00,60,NaN,NaN,NaN,NaN
1,2016-01-11 17:10:00,60,NaN,NaN,NaN,NaN
2,2016-01-11 17:20:00,50,NaN,NaN,NaN,NaN
3,2016-01-11 17:30:00,50,NaN,NaN,NaN,NaN
4,2016-01-11 17:40:00,60,NaN,NaN,NaN,NaN
5,2016-01-11 17:50:00,50,55.000000,5.477226,NaN,NaN
6,2016-01-11 18:00:00,60,55.000000,5.477226,NaN,NaN
7,2016-01-11 18:10:00,60,55.000000,5.477226,NaN,NaN
8,2016-01-11 18:20:00,60,56.666667,5.163978,NaN,NaN
9,2016-01-11 18:30:00,70,60.000000,6.324555,NaN,NaN


In [5]:
# raw calendar components
df["hour_of_day"] = df["date"].dt.hour
df["day_of_week"] = df["date"].dt.dayofweek  # Monday=0 ... Sunday=6
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# cyclical encoding at true data resolution
minute_of_day = df["date"].dt.hour * 60 + df["date"].dt.minute
df["minute_of_day_sin"] = np.sin(2 * np.pi * minute_of_day / 1440)
df["minute_of_day_cos"] = np.cos(2 * np.pi * minute_of_day / 1440)

# day-of-week is also cyclical (Sunday -> Monday wraps around)
df["day_of_week_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["day_of_week_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

df[["date", "hour_of_day", "day_of_week", "is_weekend",
    "minute_of_day_sin", "minute_of_day_cos",
    "day_of_week_sin", "day_of_week_cos"]].head(3)

,date,hour_of_day,day_of_week,is_weekend,minute_of_day_sin,minute_of_day_cos,day_of_week_sin,day_of_week_cos
0,2016-01-11 17:00:00,17,0,0,-0.965926,-0.258819,0.0,1.0
1,2016-01-11 17:10:00,17,0,0,-0.976296,-0.216440,0.0,1.0
2,2016-01-11 17:20:00,17,0,0,-0.984808,-0.173648,0.0,1.0


In [6]:
df["target_t1"] = df["Appliances"].shift(-1)
df["target_t6"] = df["Appliances"].shift(-6)

df[["date", "Appliances", "target_t1", "target_t6"]].tail(8)

,date,Appliances,target_t1,target_t6
19727,2016-05-27 16:50:00,120,110.0,420.0
19728,2016-05-27 17:00:00,110,90.0,430.0
19729,2016-05-27 17:10:00,90,100.0,NaN
19730,2016-05-27 17:20:00,100,90.0,NaN
19731,2016-05-27 17:30:00,90,270.0,NaN
19732,2016-05-27 17:40:00,270,420.0,NaN
19733,2016-05-27 17:50:00,420,430.0,NaN
19734,2016-05-27 18:00:00,430,NaN,NaN


In [7]:
print("Full range:", df["date"].min(), "→", df["date"].max())
print("Total days:", (df["date"].max() - df["date"].min()).days + 1)

# candidate boundaries — calendar-day aligned, test >= 4 full weeks
train_end   = pd.Timestamp("2016-04-18")  # exclusive start of val
val_end     = pd.Timestamp("2016-05-02")  # exclusive start of test — ~2 weeks val
test_end    = df["date"].max()

print("\nProposed boundaries:")
print("Train: start ->", train_end)
print("Val:  ", train_end, "->", val_end)
print("Test: ", val_end, "->", test_end)

for name, mask in [
    ("train", df["date"] < train_end),
    ("val",   (df["date"] >= train_end) & (df["date"] < val_end)),
    ("test",  df["date"] >= val_end),
]:
    subset = df.loc[mask]
    n_days = (subset["date"].max() - subset["date"].min()).days + 1
    print(f"{name}: {len(subset)} rows, {n_days} days, {subset['date'].min()} -> {subset['date'].max()}")

Full range: 2016-01-11 17:00:00 → 2016-05-27 18:00:00
Total days: 138

Proposed boundaries:
Train: start -> 2016-04-18 00:00:00
Val:   2016-04-18 00:00:00 -> 2016-05-02 00:00:00
Test:  2016-05-02 00:00:00 -> 2016-05-27 18:00:00
train: 14010 rows, 98 days, 2016-01-11 17:00:00 -> 2016-04-17 23:50:00
val: 2016 rows, 14 days, 2016-04-18 00:00:00 -> 2016-05-01 23:50:00
test: 3709 rows, 26 days, 2016-05-02 00:00:00 -> 2016-05-27 18:00:00


In [8]:
train_end = pd.Timestamp("2016-04-18")
val_end   = pd.Timestamp("2016-04-30")
test_end  = df["date"].max()

for name, mask in [
    ("train", df["date"] < train_end),
    ("val",   (df["date"] >= train_end) & (df["date"] < val_end)),
    ("test",  df["date"] >= val_end),
]:
    subset = df.loc[mask]
    n_days = (subset["date"].max() - subset["date"].min()).days + 1
    print(f"{name}: {len(subset)} rows, {n_days} days, {subset['date'].min()} -> {subset['date'].max()}")

train: 14010 rows, 98 days, 2016-01-11 17:00:00 -> 2016-04-17 23:50:00
val: 1728 rows, 12 days, 2016-04-18 00:00:00 -> 2016-04-29 23:50:00
test: 3997 rows, 28 days, 2016-04-30 00:00:00 -> 2016-05-27 18:00:00


In [10]:
TRAIN_END = pd.Timestamp("2016-04-18")
VAL_END   = pd.Timestamp("2016-04-30")

In [11]:
from sklearn.preprocessing import StandardScaler

# pick one feature to make the leak concrete
feature = "T_out"

train_mask = df["date"] < TRAIN_END
val_mask   = (df["date"] >= TRAIN_END) & (df["date"] < VAL_END)
test_mask  = df["date"] >= VAL_END

# WRONG: fit on entire dataset
scaler_leaky = StandardScaler()
scaler_leaky.fit(df[[feature]])
print("Leaky scaler — fit on FULL dataset:")
print(f"  mean = {scaler_leaky.mean_[0]:.4f}, std = {scaler_leaky.scale_[0]:.4f}")

# CORRECT: fit on train only
scaler_correct = StandardScaler()
scaler_correct.fit(df.loc[train_mask, [feature]])
print("\nCorrect scaler — fit on TRAIN only:")
print(f"  mean = {scaler_correct.mean_[0]:.4f}, std = {scaler_correct.scale_[0]:.4f}")

# show the actual divergence on a real training row's scaled value
sample_row = df.loc[train_mask, feature].iloc[0]
print(f"\nSame raw train value: {sample_row}")
print(f"  Scaled using LEAKY (full-data) stats : {(sample_row - scaler_leaky.mean_[0]) / scaler_leaky.scale_[0]:.4f}")
print(f"  Scaled using CORRECT (train-only) stats: {(sample_row - scaler_correct.mean_[0]) / scaler_correct.scale_[0]:.4f}")

Leaky scaler — fit on FULL dataset:
  mean = 7.4117, std = 5.3173

Correct scaler — fit on TRAIN only:
  mean = 5.7220, std = 4.1319

Same raw train value: 6.6
  Scaled using LEAKY (full-data) stats : -0.1526
  Scaled using CORRECT (train-only) stats: 0.2125


In [12]:
# columns to scale: continuous only, excludes binary/cyclical/raw-calendar
cols_to_scale = [
    "Appliances_lag_1", "Appliances_lag_2", "Appliances_lag_3", "Appliances_lag_4",
    "Appliances_lag_5", "Appliances_lag_6", "Appliances_lag_144",
    "Appliances_roll6_mean", "Appliances_roll6_std",
    "Appliances_roll18_mean", "Appliances_roll18_std",
]

scaler = StandardScaler()

train_mask = df["date"] < TRAIN_END
val_mask   = (df["date"] >= TRAIN_END) & (df["date"] < VAL_END)
test_mask  = df["date"] >= VAL_END

# fit ONLY on train rows, only on these columns
scaler.fit(df.loc[train_mask, cols_to_scale])

# transform all three partitions using the same fitted scaler
df_scaled = df.copy()
df_scaled.loc[train_mask, cols_to_scale] = scaler.transform(df.loc[train_mask, cols_to_scale])
df_scaled.loc[val_mask, cols_to_scale]   = scaler.transform(df.loc[val_mask, cols_to_scale])
df_scaled.loc[test_mask, cols_to_scale]  = scaler.transform(df.loc[test_mask, cols_to_scale])

df_scaled.loc[train_mask, cols_to_scale].describe().loc[["mean", "std"]]

,Appliances_lag_1,Appliances_lag_2,Appliances_lag_3,Appliances_lag_4,Appliances_lag_5,Appliances_lag_6,Appliances_lag_144,Appliances_roll6_mean,Appliances_roll6_std,Appliances_roll18_mean,Appliances_roll18_std
mean,3.144668e-17,1.014481e-17,-1.724741e-17,4.870206e-17,-6.189662e-17,5.987150e-17,-1.024871e-18,4.058795e-18,8.929348e-17,-1.289772e-16,-7.312096e-17
std,1.000036e+00,1.000036e+00,1.000036e+00,1.000036e+00,1.000036e+00,1.000036e+00,1.000036e+00,1.000036e+00,1.000036e+00,1.000036e+00,1.000036e+00


In [13]:
feature_columns = [
    "Appliances_lag_1", "Appliances_lag_2", "Appliances_lag_3",
    "Appliances_lag_4", "Appliances_lag_5", "Appliances_lag_6", "Appliances_lag_144",
    "Appliances_roll6_mean", "Appliances_roll6_std",
    "Appliances_roll18_mean", "Appliances_roll18_std",
    "hour_of_day", "day_of_week", "is_weekend",
    "minute_of_day_sin", "minute_of_day_cos",
    "day_of_week_sin", "day_of_week_cos",
]

# t+1 modeling dataset
t1_cols = ["date"] + feature_columns + ["target_t1"]
df_t1 = df_scaled[t1_cols].dropna(subset=feature_columns + ["target_t1"]).reset_index(drop=True)

# t+6 modeling dataset
t6_cols = ["date"] + feature_columns + ["target_t6"]
df_t6 = df_scaled[t6_cols].dropna(subset=feature_columns + ["target_t6"]).reset_index(drop=True)

print("t+1 dataset:", df_t1.shape, df_t1["date"].min(), "->", df_t1["date"].max())
print("t+6 dataset:", df_t6.shape, df_t6["date"].min(), "->", df_t6["date"].max())

# sanity: confirm no NaNs remain in either
print("\nt+1 NaN check:", df_t1.isna().sum().sum())
print("t+6 NaN check:", df_t6.isna().sum().sum())

t+1 dataset: (19590, 20) 2016-01-12 17:00:00 -> 2016-05-27 17:50:00
t+6 dataset: (19585, 20) 2016-01-12 17:00:00 -> 2016-05-27 17:00:00

t+1 NaN check: 0
t+6 NaN check: 0


In [14]:
import joblib

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_t1.to_csv(PROCESSED_DIR / "features_t1.csv", index=False)
df_t6.to_csv(PROCESSED_DIR / "features_t6.csv", index=False)
joblib.dump(scaler, PROCESSED_DIR / "scaler_train_fit.joblib")

# also persist the split boundaries as an explicit artifact — not just remembered in a notebook cell
import json
split_boundaries = {
    "train_end": str(TRAIN_END),
    "val_end": str(VAL_END),
    "test_end": str(df["date"].max()),
}
with open(PROCESSED_DIR / "split_boundaries.json", "w") as f:
    json.dump(split_boundaries, f, indent=2)

print("Saved:")
for f in PROCESSED_DIR.iterdir():
    print(" -", f.name)

Saved:
 - scaler_train_fit.joblib
 - features_t1.csv
 - split_boundaries.json
 - features_t6.csv


In [15]:
from src.features.build_features import build_features

df_raw = pd.read_csv(RAW_DATA_PATH, parse_dates=["date"])  # fresh, unmodified load
df_pipeline = build_features(df_raw)

print(df_pipeline.shape)
print(df_pipeline.columns.tolist())

(19735, 49)
['date', 'Appliances', 'lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint', 'rv1', 'rv2', 'Appliances_lag_1', 'Appliances_lag_2', 'Appliances_lag_3', 'Appliances_lag_4', 'Appliances_lag_5', 'Appliances_lag_6', 'Appliances_lag_144', 'Appliances_roll6_mean', 'Appliances_roll6_std', 'Appliances_roll18_mean', 'Appliances_roll18_std', 'hour_of_day', 'day_of_week', 'is_weekend', 'minute_of_day_sin', 'minute_of_day_cos', 'day_of_week_sin', 'day_of_week_cos', 'target_t1', 'target_t6']


In [16]:
# these should be identical if the pipeline correctly reproduces Cells 2-6
compare_cols = [
    "Appliances_lag_1", "Appliances_lag_144",
    "Appliances_roll6_mean", "Appliances_roll18_std",
    "hour_of_day", "minute_of_day_sin", "day_of_week_cos",
    "target_t1", "target_t6",
]

for col in compare_cols:
    match = df_pipeline[col].equals(df[col])
    print(f"{col}: {'MATCH' if match else 'MISMATCH'}")

Appliances_lag_1: MATCH
Appliances_lag_144: MATCH
Appliances_roll6_mean: MATCH
Appliances_roll18_std: MATCH
hour_of_day: MATCH
minute_of_day_sin: MATCH
day_of_week_cos: MATCH
target_t1: MATCH
target_t6: MATCH


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))  # repo root importable

from config.paths import RAW_DATA_PATH
from src.features.build_features import build_features

# fresh raw load — untouched by any prior cell in this kernel
df_raw = pd.read_csv(RAW_DATA_PATH, parse_dates=["date"])
df_pipeline = build_features(df_raw)

print(df_pipeline.shape)
print(df_pipeline.columns.tolist())

(19735, 49)
['date', 'Appliances', 'lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint', 'rv1', 'rv2', 'Appliances_lag_1', 'Appliances_lag_2', 'Appliances_lag_3', 'Appliances_lag_4', 'Appliances_lag_5', 'Appliances_lag_6', 'Appliances_lag_144', 'Appliances_roll6_mean', 'Appliances_roll6_std', 'Appliances_roll18_mean', 'Appliances_roll18_std', 'hour_of_day', 'day_of_week', 'is_weekend', 'minute_of_day_sin', 'minute_of_day_cos', 'day_of_week_sin', 'day_of_week_cos', 'target_t1', 'target_t6']


In [2]:
# original notebook-native build, for comparison — same logic as Cells 2-6, done manually
df_manual = df_raw.sort_values("date").reset_index(drop=True)

LAG_STEPS = [1, 2, 3, 4, 5, 6, 144]
for lag in LAG_STEPS:
    df_manual[f"Appliances_lag_{lag}"] = df_manual["Appliances"].shift(lag)

ROLLING_WINDOWS = [6, 18]
for window in ROLLING_WINDOWS:
    df_manual[f"Appliances_roll{window}_mean"] = df_manual["Appliances"].rolling(window=window).mean()
    df_manual[f"Appliances_roll{window}_std"] = df_manual["Appliances"].rolling(window=window).std()

df_manual["hour_of_day"] = df_manual["date"].dt.hour
df_manual["day_of_week"] = df_manual["date"].dt.dayofweek
df_manual["is_weekend"] = (df_manual["day_of_week"] >= 5).astype(int)

minute_of_day = df_manual["date"].dt.hour * 60 + df_manual["date"].dt.minute
df_manual["minute_of_day_sin"] = np.sin(2 * np.pi * minute_of_day / 1440)
df_manual["minute_of_day_cos"] = np.cos(2 * np.pi * minute_of_day / 1440)
df_manual["day_of_week_sin"] = np.sin(2 * np.pi * df_manual["day_of_week"] / 7)
df_manual["day_of_week_cos"] = np.cos(2 * np.pi * df_manual["day_of_week"] / 7)

df_manual["target_t1"] = df_manual["Appliances"].shift(-1)
df_manual["target_t6"] = df_manual["Appliances"].shift(-6)

print(df_manual.shape)

(19735, 49)


In [3]:
new_cols = [c for c in df_pipeline.columns if c not in df_raw.columns]

all_match = True
for col in new_cols:
    match = df_pipeline[col].equals(df_manual[col])
    status = "MATCH" if match else "MISMATCH"
    if not match:
        all_match = False
    print(f"{col}: {status}")

print("\nALL COLUMNS MATCH:", all_match)

Appliances_lag_1: MATCH
Appliances_lag_2: MATCH
Appliances_lag_3: MATCH
Appliances_lag_4: MATCH
Appliances_lag_5: MATCH
Appliances_lag_6: MATCH
Appliances_lag_144: MATCH
Appliances_roll6_mean: MATCH
Appliances_roll6_std: MATCH
Appliances_roll18_mean: MATCH
Appliances_roll18_std: MATCH
hour_of_day: MATCH
day_of_week: MATCH
is_weekend: MATCH
minute_of_day_sin: MATCH
minute_of_day_cos: MATCH
day_of_week_sin: MATCH
day_of_week_cos: MATCH
target_t1: MATCH
target_t6: MATCH

ALL COLUMNS MATCH: True


In [4]:
# 1. Confirm: does the train partition actually contain the burn-in NaN rows?
cols_to_scale = [
    "Appliances_lag_1", "Appliances_lag_2", "Appliances_lag_3", "Appliances_lag_4",
    "Appliances_lag_5", "Appliances_lag_6", "Appliances_lag_144",
    "Appliances_roll6_mean", "Appliances_roll6_std",
    "Appliances_roll18_mean", "Appliances_roll18_std",
]

train_mask = df["date"] < TRAIN_END

nan_counts = df.loc[train_mask, cols_to_scale].isna().sum()
print("NaN counts in train partition, per column:")
print(nan_counts)
print("\nTotal rows in train partition:", train_mask.sum())

# 2. Try fitting a scaler directly on the RAW train partition (including NaN rows)
#    and see what actually happens
from sklearn.preprocessing import StandardScaler

test_scaler = StandardScaler()
try:
    test_scaler.fit(df.loc[train_mask, cols_to_scale])
    print("\nfit() SUCCEEDED — no error raised")
    print("mean:", test_scaler.mean_)
except ValueError as e:
    print("\nfit() RAISED ValueError:")
    print(e)

NameError: name 'df' is not defined

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

from config.paths import RAW_DATA_PATH
from src.features.build_features import build_features

df = build_features(pd.read_csv(RAW_DATA_PATH, parse_dates=["date"]))

TRAIN_END = pd.Timestamp("2016-04-18")

cols_to_scale = [
    "Appliances_lag_1", "Appliances_lag_2", "Appliances_lag_3", "Appliances_lag_4",
    "Appliances_lag_5", "Appliances_lag_6", "Appliances_lag_144",
    "Appliances_roll6_mean", "Appliances_roll6_std",
    "Appliances_roll18_mean", "Appliances_roll18_std",
]

train_mask = df["date"] < TRAIN_END

nan_counts = df.loc[train_mask, cols_to_scale].isna().sum()
print("NaN counts in train partition, per column:")
print(nan_counts)
print("\nTotal rows in train partition:", train_mask.sum())

from sklearn.preprocessing import StandardScaler

test_scaler = StandardScaler()
try:
    test_scaler.fit(df.loc[train_mask, cols_to_scale])
    print("\nfit() SUCCEEDED — no error raised")
    print("mean:", test_scaler.mean_)
except ValueError as e:
    print("\nfit() RAISED ValueError:")
    print(e)

NaN counts in train partition, per column:
Appliances_lag_1            1
Appliances_lag_2            2
Appliances_lag_3            3
Appliances_lag_4            4
Appliances_lag_5            5
Appliances_lag_6            6
Appliances_lag_144        144
Appliances_roll6_mean       5
Appliances_roll6_std        5
Appliances_roll18_mean     17
Appliances_roll18_std      17
dtype: int64

Total rows in train partition: 14010

fit() SUCCEEDED — no error raised
mean: [99.05917624 99.06125071 99.06332548 99.06611452 99.06961799 99.07312197
 98.98384538 99.06997501 39.40131356 99.07735614 55.02976468]


In [7]:
import pandas as pd

from config.paths import RAW_DATA_PATH
from src.pipeline import run_pipeline

# Load raw data
df_raw = pd.read_csv(
    RAW_DATA_PATH,
    parse_dates=["date"],
)

# Run complete pipeline
result = run_pipeline(df_raw)

# Check shapes
print("train_t1:", result.train_t1.shape)
print("val_t1:  ", result.val_t1.shape)
print("test_t1: ", result.test_t1.shape)
print("train_t6:", result.train_t6.shape)
print("val_t6:  ", result.val_t6.shape)
print("test_t6: ", result.test_t6.shape)

train_t1: (13866, 20)
val_t1:   (1728, 20)
test_t1:  (3996, 20)
train_t6: (13866, 20)
val_t6:   (1728, 20)
test_t6:  (3991, 20)


In [8]:
datasets = {
    "train_t1": result.train_t1,
    "val_t1": result.val_t1,
    "test_t1": result.test_t1,
    "train_t6": result.train_t6,
    "val_t6": result.val_t6,
    "test_t6": result.test_t6,
}

for name, data in datasets.items():
    print(f"{name}: NaNs = {data.isna().sum().sum()}")

train_t1: NaNs = 0
val_t1: NaNs = 0
test_t1: NaNs = 0
train_t6: NaNs = 0
val_t6: NaNs = 0
test_t6: NaNs = 0


In [9]:
for partition in ["train", "val", "test"]:
    t1_rows = len(getattr(result, f"{partition}_t1"))
    t6_rows = len(getattr(result, f"{partition}_t6"))

    print(
        f"{partition}: "
        f"t1={t1_rows}, "
        f"t6={t6_rows}, "
        f"difference={t1_rows - t6_rows}"
    )

train: t1=13866, t6=13866, difference=0
val: t1=1728, t6=1728, difference=0
test: t1=3996, t6=3991, difference=5


In [10]:
import pandas as pd
import joblib
import json
from pathlib import Path

from config.paths import RAW_DATA_PATH
from config.features import SPLIT_TRAIN_END, SPLIT_VAL_END
from src.pipeline import run_pipeline

df_raw = pd.read_csv(RAW_DATA_PATH, parse_dates=["date"])
result = run_pipeline(df_raw)

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"

# t+1: combine train/val/test back for storage, OR store separately — 
# separately is more honest about what the split actually is
result.train_t1.to_csv(PROCESSED_DIR / "train_t1.csv", index=False)
result.val_t1.to_csv(PROCESSED_DIR / "val_t1.csv", index=False)
result.test_t1.to_csv(PROCESSED_DIR / "test_t1.csv", index=False)

result.train_t6.to_csv(PROCESSED_DIR / "train_t6.csv", index=False)
result.val_t6.to_csv(PROCESSED_DIR / "val_t6.csv", index=False)
result.test_t6.to_csv(PROCESSED_DIR / "test_t6.csv", index=False)

joblib.dump(result.scaler, PROCESSED_DIR / "scaler_train_fit.joblib")

split_boundaries = {"train_end": SPLIT_TRAIN_END, "val_end": SPLIT_VAL_END}
with open(PROCESSED_DIR / "split_boundaries.json", "w") as f:
    json.dump(split_boundaries, f, indent=2)

# remove the old, pre-pipeline combined files — they're now superseded
old_files = [PROCESSED_DIR / "features_t1.csv", PROCESSED_DIR / "features_t6.csv"]
for f in old_files:
    if f.exists():
        f.unlink()
        print("removed superseded file:", f.name)

print("\nSaved (pipeline-verified):")
for f in sorted(PROCESSED_DIR.iterdir()):
    print(" -", f.name)

removed superseded file: features_t1.csv
removed superseded file: features_t6.csv

Saved (pipeline-verified):
 - scaler_train_fit.joblib
 - split_boundaries.json
 - test_t1.csv
 - test_t6.csv
 - train_t1.csv
 - train_t6.csv
 - val_t1.csv
 - val_t6.csv
